# CoSQA E5-base-v2 + HyDE

This notebook is the ordered entry point for the AZH-515 experiment. It evaluates
real HyDE query expansion on the pinned `CoIR-Retrieval/cosqa` test qrels with
the same E5 first-stage retriever, complete corpus, 1,000-document candidate
depth, and official
COIR evaluator as the baseline.

The default execution mode is a small **real-model smoke run**. Smoke output
proves wiring only and is not benchmark evidence. Set `E5_HYDE_MODE=benchmark`
for the complete declared test split and corpus.

The generator is `google/flan-t5-base` at the immutable revision recorded in
`e5_hyde.py`. It receives only the original query and generates one
code-oriented hypothesis. The original query IDs remain the evaluation keys;
qrels, answers, labels, and target documents never enter generation.

In [ ]:
# Install code-retrieval/requirements.txt once before running this notebook.
# In Colab, uncomment the next line and restart the runtime if pip updates torch/numpy.
# %pip install -r ../requirements.txt

from pathlib import Path
import gc
import hashlib
import json
import os
import sys
import time

workspace_root = Path.cwd()
while workspace_root != workspace_root.parent and not (workspace_root / 'code-retrieval' / 'src').exists():
    workspace_root = workspace_root.parent
project_dir = workspace_root / 'code-retrieval'
if not (project_dir / 'src').exists():
    project_dir = Path.cwd()
    workspace_root = project_dir.parent

os.chdir(project_dir)
sys.path.insert(0, str(project_dir / 'src'))

from e5_hyde import (
    HyDEConfig,
    HyDEGenerator,
    E5Encoder,
    build_expanded_queries,
    build_rrf_rankings,
    build_hyde_result,
    environment_metadata,
    expected_hyde_cache_metadata,
    hyde_cache_paths,
    hyde_run_identity,
    load_valid_json_cache,
    rank_with_faiss,
    save_json_cache,
    validate_hypotheses,
)
from e5_baseline import (
    evaluate_ndcg_at_10,
    load_cosqa,
    load_valid_embedding_cache,
    package_versions,
    save_embedding_cache,
    select_run_data,
    set_seed,
)

print('Project directory:', project_dir.resolve())
print('Python:', sys.version.split()[0])

## 1. Environment and explicit controls

All controls that affect comparable runs are declared here. Dataset and model
revisions are immutable; the generator prompt, revision, stopping behavior,
batching, combination strategy, and empty-output policy are part of the run
identity and result metadata.

In [ ]:
RUN_MODE = os.environ.get('E5_HYDE_MODE', 'smoke').strip().lower()
if RUN_MODE not in {'smoke', 'benchmark'}:
    raise ValueError('E5_HYDE_MODE must be smoke or benchmark')

batch_size = int(os.environ.get('E5_HYDE_BATCH_SIZE', '32' if RUN_MODE == 'smoke' else '64'))
generation_batch_size = int(os.environ.get('E5_HYDE_GENERATION_BATCH_SIZE', '8'))
torch_threads = int(os.environ.get('E5_HYDE_TORCH_THREADS', '0'))
if torch_threads > 0:
    import torch
    torch.set_num_threads(torch_threads)
    torch.set_num_interop_threads(1)

config = HyDEConfig(
    run_mode=RUN_MODE,
    batch_size=batch_size,
    generation_batch_size=generation_batch_size,
    combination_strategy=os.environ.get('E5_HYDE_COMBINATION_STRATEGY', 'original_hyde_rrf'),
    max_new_tokens=int(os.environ.get('E5_HYDE_MAX_NEW_TOKENS', '32')),
    rrf_k=5,
    rrf_hyde_weight=0.5,
    cache_dir='artifacts/e5_hyde/cache',
    artifact_dir='artifacts/e5_hyde',
)
set_seed(config.seed)

print(json.dumps(config.as_dict(), indent=2, sort_keys=True))
print('Installed packages:')
print(json.dumps(package_versions([
    'coir-eval', 'datasets', 'faiss-cpu', 'numpy', 'pytrec-eval-terrier',
    'sentence-transformers', 'torch', 'transformers'
]), indent=2, sort_keys=True))
print('Hardware/runtime:')
print(json.dumps(environment_metadata(config, repo_root=workspace_root), indent=2, sort_keys=True, default=str))

## 2. Load and inspect the real CosQA schema

The loader reads the separate `corpus`, `queries`, and `default/test`
configurations from the same pinned dataset revision. It adapts only after
checking the observed columns and preserves source IDs. Titles are excluded to
match the paper-compatible COIR text-only path.

In [ ]:
data = load_cosqa(config)
print(json.dumps(data.schema, indent=2, sort_keys=True, default=str))
print({
    'corpus_count': len(data.corpus),
    'query_count_with_test_qrels': len(data.queries),
    'qrels_query_count': len(data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in data.qrels.values()),
})
print('Corpus example:', next(iter(data.corpus.items())))
print('Query example:', next(iter(data.queries.items())))
print('Qrels example:', next(iter(data.qrels.items())))

## 3. Select evidence level

Smoke mode keeps the first few labeled queries and a small corpus, then adds
each selected query's judged documents so the evaluator receives a valid
non-empty ranking problem. Benchmark mode uses every test-qrels query and every
corpus document.

In [ ]:
run_data = select_run_data(data, config)
if config.run_mode == 'benchmark' and (
    len(run_data.queries) != len(data.queries)
    or len(run_data.corpus) != len(data.corpus)
):
    raise RuntimeError(
        'benchmark mode must use the complete declared CosQA test queries and corpus'
    )

print(json.dumps({
    'run_mode': config.run_mode,
    'query_count': len(run_data.queries),
    'corpus_count': len(run_data.corpus),
    'qrels_query_count': len(run_data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in run_data.qrels.values()),
    'exclusions': run_data.exclusions,
}, indent=2))

## 4. Run identity and cache paths

HyDE hypotheses, expanded queries, embeddings, and rankings are cached
separately. Every cache is tied to the full experiment configuration, immutable
revisions, repository commit, notebook hash, and ordered source IDs.

In [ ]:
notebook_path = project_dir / 'notebooks' / 'e5_hyde_experiment.ipynb'
notebook_sha256 = hashlib.sha256(notebook_path.read_bytes()).hexdigest()
identity = hyde_run_identity(
    config,
    repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
)
paths = hyde_cache_paths(config, identity)
print('Run identity:', identity)
print('Notebook SHA-256:', notebook_sha256)
print(json.dumps({name: str(path) for name, path in paths.items()}, indent=2))

## 5. Generate and cache one real HyDE hypothesis per query

The generator receives only original query text. Generation is deterministic
(`temperature=0`, no sampling), stops on the model EOS/pad behavior, and uses
the explicit `original_query_fallback` policy if decoding yields only special
tokens. This preserves the original query and does not invent synthetic text.
A valid cache is reused only
when its exact metadata matches the current run.

In [ ]:
query_ids = list(run_data.queries)
hypotheses_metadata = expected_hyde_cache_metadata(
    config,
    identity=identity,
    kind='hypotheses',
    ids=query_ids,
    repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
)
hypotheses = load_valid_json_cache(
    paths['hypotheses'],
    paths['hypotheses_metadata'],
    hypotheses_metadata,
)
generation_seconds = 0.0
if hypotheses is None:
    generation_started = time.perf_counter()
    generator = HyDEGenerator(config)
    hypotheses = generator.generate(run_data.queries)
    generation_seconds = time.perf_counter() - generation_started
    save_json_cache(
        paths['hypotheses'],
        paths['hypotheses_metadata'],
        hypotheses,
        hypotheses_metadata,
    )
    del generator
    gc.collect()
else:
    hypotheses = validate_hypotheses(
        run_data.queries,
        hypotheses,
        empty_hypothesis_behavior=config.empty_hypothesis_behavior,
    )

print('Hypotheses:', len(hypotheses), 'seconds:', round(generation_seconds, 3))
print('First hypothesis:', next(iter(hypotheses.items())))

## 6. Build and cache the expanded query representation

The original query remains the evaluation key. The default benchmark
combination uses weighted reciprocal-rank fusion of the original E5 run and
the HyDE-only E5 run; the legacy concatenation strategy remains available.

In [ ]:
expanded_metadata = expected_hyde_cache_metadata(
    config,
    identity=identity,
    kind='expanded_queries',
    ids=query_ids,
    repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
)
expanded_queries = load_valid_json_cache(
    paths['expanded_queries'],
    paths['expanded_queries_metadata'],
    expanded_metadata,
)
if expanded_queries is None:
    expanded_queries = build_expanded_queries(
        run_data.queries,
        hypotheses,
        strategy='original_plus_hypothesis',
    )
    save_json_cache(
        paths['expanded_queries'],
        paths['expanded_queries_metadata'],
        expanded_queries,
        expanded_metadata,
    )
else:
    expected_expanded = build_expanded_queries(
        run_data.queries,
        hypotheses,
        strategy='original_plus_hypothesis',
    )
    if expanded_queries != expected_expanded:
        raise RuntimeError('expanded query cache content does not match current hypotheses')

print('Expanded queries:', len(expanded_queries))
print('First expanded query:', next(iter(expanded_queries.items())))

## 7. Encode the corpus with E5

E5 receives `passage: ` for corpus text and `query: ` for query text.
Sentence Transformers performs the documented pooling; embeddings are normalized
and truncated to 512 tokens. Corpus embeddings are independent of HyDE text but
remain in this experiment's separate cache namespace.

In [ ]:
encoder = E5Encoder(config)
corpus_ids = list(run_data.corpus)
corpus_texts = [run_data.corpus[doc_id]['text'] for doc_id in corpus_ids]
corpus_metadata = expected_hyde_cache_metadata(
    config,
    identity=identity,
    kind='corpus_embeddings',
    ids=corpus_ids,
    repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
)
corpus_embeddings = load_valid_embedding_cache(
    paths['corpus_embeddings'],
    paths['corpus_metadata'],
    corpus_metadata,
)
if corpus_embeddings is None:
    corpus_started = time.perf_counter()
    corpus_embeddings = encoder.encode_corpus(corpus_texts)
    corpus_seconds = time.perf_counter() - corpus_started
    save_embedding_cache(
        paths['corpus_embeddings'],
        paths['corpus_metadata'],
        corpus_embeddings,
        corpus_metadata,
    )
else:
    corpus_seconds = 0.0
print('Corpus embeddings:', corpus_embeddings.shape, 'seconds:', round(corpus_seconds, 3))

## 8. Encode the query representations

For `original_hyde_rrf`, the original query and generated hypothesis are encoded
separately so their exact candidate rankings can be fused. The original query IDs
and qrels are passed unchanged to the evaluator.

In [ ]:
query_started = time.perf_counter()
if config.combination_strategy == 'original_hyde_rrf':
    original_query_metadata = expected_hyde_cache_metadata(
        config, identity=identity, kind='original_query_embeddings', ids=query_ids,
        repo_root=workspace_root, notebook_sha256=notebook_sha256,
    )
    hypothesis_metadata = expected_hyde_cache_metadata(
        config, identity=identity, kind='hypothesis_embeddings', ids=query_ids,
        repo_root=workspace_root, notebook_sha256=notebook_sha256,
    )
    original_query_embeddings = load_valid_embedding_cache(
        paths['original_query_embeddings'], paths['original_query_metadata'],
        original_query_metadata,
    )
    hypothesis_embeddings = load_valid_embedding_cache(
        paths['hypothesis_embeddings'], paths['hypothesis_metadata'],
        hypothesis_metadata,
    )
    if original_query_embeddings is None:
        original_query_embeddings = encoder.encode_queries([run_data.queries[qid] for qid in query_ids])
        save_embedding_cache(paths['original_query_embeddings'], paths['original_query_metadata'], original_query_embeddings, original_query_metadata)
    if hypothesis_embeddings is None:
        hypothesis_embeddings = encoder.encode_queries([hypotheses[qid] for qid in query_ids])
        save_embedding_cache(paths['hypothesis_embeddings'], paths['hypothesis_metadata'], hypothesis_embeddings, hypothesis_metadata)
    query_metadata = {'original': original_query_metadata, 'hypothesis': hypothesis_metadata}
    query_embeddings = None
else:
    expanded_texts = [expanded_queries[query_id] for query_id in query_ids]
    query_metadata = expected_hyde_cache_metadata(
        config, identity=identity, kind='query_embeddings', ids=query_ids,
        repo_root=workspace_root, notebook_sha256=notebook_sha256,
    )
    query_embeddings = load_valid_embedding_cache(
        paths['query_embeddings'], paths['query_metadata'], query_metadata,
    )
    if query_embeddings is None:
        query_embeddings = encoder.encode_queries(expanded_texts)
        save_embedding_cache(paths['query_embeddings'], paths['query_metadata'], query_embeddings, query_metadata)
query_seconds = time.perf_counter() - query_started
print('Query representations encoded; seconds:', round(query_seconds, 3))

## 9. Exact first-stage retrieval

The ranking uses an exact Faiss `IndexFlatIP` over normalized embeddings,
equivalent to cosine similarity. Rankings cannot introduce documents outside
the selected corpus and use the shared candidate depth of 1,000. RRF retains
the union of the original and HyDE candidate runs.

In [ ]:
ranking_started = time.perf_counter()
ranking_ids = query_ids + ['__corpus__'] + corpus_ids
ranking_metadata = expected_hyde_cache_metadata(
    config, identity=identity, kind='rankings', ids=ranking_ids,
    repo_root=workspace_root, notebook_sha256=notebook_sha256,
)
rankings = load_valid_json_cache(paths['rankings'], paths['rankings_metadata'], ranking_metadata)
if rankings is None:
    if config.combination_strategy == 'original_hyde_rrf':
        original_ranking_metadata = expected_hyde_cache_metadata(
            config, identity=identity, kind='original_rankings', ids=ranking_ids,
            repo_root=workspace_root, notebook_sha256=notebook_sha256,
        )
        hypothesis_ranking_metadata = expected_hyde_cache_metadata(
            config, identity=identity, kind='hypothesis_rankings', ids=ranking_ids,
            repo_root=workspace_root, notebook_sha256=notebook_sha256,
        )
        original_rankings = load_valid_json_cache(paths['original_rankings'], paths['original_rankings_metadata'], original_ranking_metadata)
        hypothesis_rankings = load_valid_json_cache(paths['hypothesis_rankings'], paths['hypothesis_rankings_metadata'], hypothesis_ranking_metadata)
        if original_rankings is None:
            original_rankings = rank_with_faiss(original_query_embeddings, corpus_embeddings, query_ids, corpus_ids, top_k=config.candidate_depth)
            save_json_cache(paths['original_rankings'], paths['original_rankings_metadata'], original_rankings, original_ranking_metadata, sort_keys=False)
        if hypothesis_rankings is None:
            hypothesis_rankings = rank_with_faiss(hypothesis_embeddings, corpus_embeddings, query_ids, corpus_ids, top_k=config.candidate_depth)
            save_json_cache(paths['hypothesis_rankings'], paths['hypothesis_rankings_metadata'], hypothesis_rankings, hypothesis_ranking_metadata, sort_keys=False)
        rankings = build_rrf_rankings(
            original_rankings, hypothesis_rankings,
            rrf_k=config.rrf_k, hyde_weight=config.rrf_hyde_weight,
        )
    else:
        rankings = rank_with_faiss(query_embeddings, corpus_embeddings, query_ids, corpus_ids, top_k=config.candidate_depth)
    save_json_cache(paths['rankings'], paths['rankings_metadata'], rankings, ranking_metadata)
ranking_seconds = time.perf_counter() - ranking_started
print('Ranking queries:', len(rankings))
print('First ranking:', next(iter(rankings.items())))

## 10. Official COIR evaluation

COIR's `EvaluateRetrieval` implementation uses the project's
`pytrec-eval-terrier` dependency and returns `nDCG@10` along with secondary
metrics. The aggregate is measured from the current rankings and qrels; it is
never manually entered.

In [ ]:
evaluation_started = time.perf_counter()
metric = evaluate_ndcg_at_10(run_data.qrels, rankings, cutoff=10)
evaluation_seconds = time.perf_counter() - evaluation_started
print(json.dumps(metric, indent=2, sort_keys=True))

## 11. Persist result and provenance

A smoke result is explicitly marked `smoke` and
`benchmark_evidence: false`. Only a complete benchmark-mode run over the
declared query/corpus population is marked `benchmark`.

In [ ]:
environment = environment_metadata(config, repo_root=workspace_root)
result = build_hyde_result(
    config,
    data,
    run_data,
    metric,
    identity=identity,
    environment=environment,
    timings={
        'generation': generation_seconds,
        'corpus_encoding': corpus_seconds,
        'query_encoding': query_seconds,
        'ranking': ranking_seconds,
        'evaluation': evaluation_seconds,
    },
    hypothesis_count=len(hypotheses),
    repo_root=workspace_root,
    notebook_sha256=notebook_sha256,
)
result['cache_paths'] = {name: str(path) for name, path in paths.items()}
result['cache_metadata'] = {
    'hypotheses': hypotheses_metadata,
    'expanded_queries': expanded_metadata,
    'corpus': corpus_metadata,
    'queries': query_metadata,
    'rankings': ranking_metadata,
}
from e5_baseline import write_result_artifacts
artifact_paths = write_result_artifacts(
    config,
    result,
    metadata={
        'result': result,
        'cache_metadata': result['cache_metadata'],
    },
)
print(json.dumps({
    'status': result['status'],
    'benchmark_evidence': result['benchmark_evidence'],
    'nDCG@10': result['ndcg_at_10'],
    'system_id': result['system_id'],
    'tqe': result['tqe'],
    'query_count': result['query_count'],
    'corpus_count': result['corpus_count'],
    'artifact_paths': artifact_paths,
}, indent=2))

## Interpretation boundary

This issue delivers the E5 + HyDE experiment only. Smoke metrics are useful for
checking data/model/evaluator wiring but must not be interpreted as benchmark
performance. Compare the HyDE result with AZH-514's complete `benchmark`
artifact only after both runs have `status: benchmark`.